In [ ]:
from pinoco import (ODEEquation, LoadedTrajectoryDataset, SimulatedTrajectoryDataset, SubtrajectoryDataset, 
                    SubtrajectoryView, PINNTrainDataset, TorchODEResidual, make_pinn_dataloader)

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Subset

from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from collections import defaultdict

from solis_nn import SOLIS, get_global_signal_stats, get_global_time_scale, MultitrajectoryIPINN, SimpleIPINN, save_model_package, load_checkpoint
from solis_training import train_epoch, train_epoch_ipinn, train_epoch_simple_ipinn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
print(f"Using device: {device}")

In [ ]:
train_ds = LoadedTrajectoryDataset(
    "./datasets/twotank_data_train.csv",
    traj_id_col="traj_id",
    t_col="t",
    y_cols=("y", "v"),             
    exo_cols={"u": "u"},           
    sort_by_time=True,
    device="cpu",
    dtype=torch.float32,
)

# Numerical v calculation 
for d in train_ds:
    yy = d["y"][:,0]
    tt = d["t"][:,0]
    ddy = ((yy[1:]-yy[:-1])/(tt[1:]-tt[:-1]))  # approximate velocity
    d["y"][:-1,1] = ddy
    d["y"][-1,1] = ddy[-1]  # last velocity same as second last

subds_train = SubtrajectoryView(
    SubtrajectoryDataset(
        train_ds,
        subseq_len=200,
        return_relative_time=False,
        also_return_t_abs=True,
    )
)

pinn_train = PINNTrainDataset(
    subds_train,
    collocation_frac=1.,
    num_data_per_traj=50,
    disjoint_sets=False,
    include_exogenous=True,
    global_collocation=False,
    use_importance_sampling=True,
    importance_u_weight=5,
    importance_y_weight=5,
)

loader = make_pinn_dataloader(
    pinn_train,
    n_traj_samples=4,
    n_data_samples=1000000,
    seed=123,
    shuffle_trajs=False
)

## Simple Visualizations

In [ ]:
plt.figure(figsize=(10,6))
for b in loader:
    print(b["y0"].shape)
    print(b["t_col"].shape)
    print(b["y_data"].shape)
    print(b["t_data"].shape)
    print(b["traj_id"])

    for traj_id in range(b["y0"].shape[0]):
        plt.plot(b["t_col"][traj_id].numpy(), np.zeros_like((b["t_col"][traj_id].numpy())))
        plt.scatter(b["t_data"][traj_id].numpy(), b["y_data"][traj_id,:,0].numpy(), color='r',s=2)
        # plt.scatter(b["t_data"][traj_id].numpy(), b["y_data"][traj_id,:,1].numpy(), color='g',s=2)
        plt.plot(b["t_col"][traj_id,0].numpy(), b["y0"][traj_id,:,0].numpy(), color='r', marker='o', markersize=8)
        # plt.plot(b["t_col"][traj_id,0].numpy(), b["y0"][traj_id,:,1].numpy(), color='g', marker='o', markersize=8)
        plt.plot(b["t_col"][traj_id].numpy(), b["exo_col"]["u"][traj_id].numpy(), linestyle='--')

    break
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

# Pick one trajectory and its sampled training set
dset = train_ds[0]
pdset = pinn_train[0]

# Continuous ODE solution
ax.plot(dset["t"].cpu(), dset["y"][:,0].cpu(),
        label="y(t) Solution", color="C0", linewidth=2, alpha=0.6)

# ax.plot(dset["t"].cpu(), dset["y"][:,1].cpu(),
#         label="y(t) Solution", color="C0", linewidth=2, alpha=0.6)

ax.plot(dset["t"].cpu(), dset["exo"]["u"].cpu(),
        label="u(t)", color="C4", linewidth=2, alpha=0.6)

# Data points
ax.scatter(pdset["t_data"].cpu(), pdset["y_data"][:,0].cpu(),
           label="Data Points", color="C1", marker="o", edgecolors="k")

# ax.scatter(pdset["t_data"].cpu(), pdset["y_data"][:,1].cpu(),
#            label="Data Points", color="C1", marker="o", edgecolors="k")

# Initial condition
ax.scatter(pdset["t_col"][0].cpu(), pdset["y0"][0].cpu(),
           label="Initial Value", color="C2", marker="s", s=80, edgecolors="k")

# Collocation points (y=0 just for visualization)
ax.scatter(pdset["t_col"].cpu(),
           torch.zeros_like(pdset["t_col"]).cpu(),
           label="Collocation Points", color="C3", marker="x")

# Styling
ax.set_xlabel("Time $t$")
ax.set_ylabel("$y(t)$")
ax.set_title(f"Trajectory")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

plt.tight_layout()
plt.show()

## Training

In [ ]:
global_t_min, global_t_scale = get_global_time_scale(pinn_train)
global_t_min = torch.tensor(global_t_min)
global_t_scale = torch.tensor(global_t_scale)

stats = get_global_signal_stats(pinn_train)

In [ ]:
# 2nd order hypothesis
plant_eq_y  = "Eq(D(y,1), v)"
plant_eq_dy = "Eq(D(v,1), -k*y - d*v + g*u)"

eq_pred = ODEEquation(
    eqs = [
        plant_eq_y,
        plant_eq_dy,
    ],  
    params={}, 
    dependent_variables=["y", "v"],
    exogenous_functions=["u", "k", "d", "g"],           
)

### PI2NDi Training

In [ ]:
# Models and settings
cfg_solis = dict(
                    x_dim=3, use_u_in_sol_net=True,
                    include_abs_time=False, use_relative_time=True,
                    use_moe=False, poly_order=1, ensure_positive_wn=True,
                    sol_net_layers=4, sol_net_hidden_dim=256, param_net_hidden_dim=128,
                    use_fourier_time=False, include_u_in_params=True,
                    num_trajectories=len(pinn_train),
                    use_input_normalization=True,
                    context_hidden_dim=64, context_dim=32,
)

solis = SOLIS(**cfg_solis)

solis.set_time_bounds(global_t_min.item(), (global_t_min + global_t_scale).item())

plant_params = list(solis.y_net.parameters())
if getattr(solis, "traj_emb", None) is not None:
    plant_params += list(solis.traj_emb.parameters())

if solis.use_moe:
    param_params = (
        list(solis.param_feat.parameters()) +
        list(solis.gating_net.parameters()) +
        [p for exp in solis.experts for p in exp.parameters()]
    )
else:
    param_params = list(solis.param_feat.parameters()) + list(solis.param_head.parameters())

use_velocity_data = False
use_velocity_ic   = False

if use_velocity_data:
    data_dim = 2
    print("Using position and velocity data for training. Data dim =", data_dim)
else:
    data_dim = 1
    print("Using position-only data for training. Data dim =", data_dim)

if use_velocity_ic:
    ic_dim = 2
    print("Using position and velocity in initial conditions. IC dim =", ic_dim)
else:
    ic_dim = 1
    print("Using position-only in initial conditions. IC dim =", ic_dim)

# Optimizers 
optimizer_plant  = torch.optim.AdamW(plant_params, lr=5e-4, weight_decay=5e-5)
optimizer_params = torch.optim.AdamW(param_params, lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_plant  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_plant, mode="min", factor=0.5, patience=50)
scheduler_params = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_params, mode="min", factor=0.5, patience=50)

# Tensorboard
# now_str = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
# writer = SummaryWriter(log_dir=f"runs/PI2NDi-Phint-2nd{now_str}", flush_secs=3)
writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 5_000

# Define Phase Durations
phase1_epochs = 50  
phase2_epochs = 50  
epoch_period = phase1_epochs + phase2_epochs 

# Starts at 1.0, hits 0 at 5% of total epochs
phint_cutoff_ratio = 0.05

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    phase_epoch = (epoch-1) % epoch_period

    # --------------------------------------
    # 1. Determine Phase & Hyperparameters
    # --------------------------------------
    if phase_epoch < phase1_epochs:
        current_phase = 1
        phase_desc = "Phase 1: Solution Learning"
        
    elif phase_epoch < (phase1_epochs + phase2_epochs):
        current_phase = 2
        phase_desc = "Phase 2: Parameter Identification"
    
    change_phase = not (current_phase == prev_phase)

    # hint_decay = max(0.0, 1.0 - epoch / (epochs * phint_cutoff_ratio))
    hint_decay = 0
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch(
        model=solis,
        residual=residual,
        eq=eq_pred,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer_plant=optimizer_plant,
        optimizer_params=optimizer_params,
        criterion=criterion,
        phase=current_phase,  
        change_phase=change_phase,
        scheduler_plant=scheduler_plant,
        scheduler_params=scheduler_params,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
        lambda_gate_reg=0.01,
        lambda_phy_hint=.1 * hint_decay,
        lambda_tv=2,
        H_horizon=1,
        lambda_step=0.01,
        random_window="large"
    )

    prev_phase = current_phase

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    # Update tqdm status bar
    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"{phase_desc}")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

### IPINN-M Training

In [ ]:
cfg_ipinn_m = dict(
                    x_dim=3, use_u_in_sol_net=True,
                    include_abs_time=False, use_relative_time=True,
                    use_moe=False, poly_order=1, ensure_positive_wn=True,
                    sol_net_layers=4, sol_net_hidden_dim=256, param_net_hidden_dim=128,
                    use_fourier_time=False, include_u_in_params=True,
                    traj_emb_dim=0, num_trajectories=len(pinn_train),
                    use_input_normalization=True,
                    context_hidden_dim=64, context_dim=32,
)

ipinn_m = MultitrajectoryIPINN(**cfg_ipinn_m)

use_velocity_data = False
use_velocity_ic   = False

if use_velocity_data:
    print("Using position and velocity data for training.")
    data_dim = 2
else:
    print("Using position-only data for training.")
    data_dim = 1

if use_velocity_ic:
    print("Using position and velocity in initial conditions.")
    ic_dim = 2
else:
    print("Using position-only in initial conditions.")
    ic_dim = 1

# Optimizers 
optimizer_ipinn_m  = torch.optim.AdamW(ipinn_m.parameters(), lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_ipinn_m  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ipinn_m, mode="min", factor=0.5, patience=50)
writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 5_000

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch_ipinn(
        model=ipinn_m,
        residual=residual,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer=optimizer_ipinn_m,
        criterion=criterion,
        scheduler=scheduler_ipinn_m,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
    )

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"[Training]")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

### IPINN Training

In [ ]:
cfg_ipinn_simple = dict(
                    x_dim=3, use_u_in_sol_net=True,
                    include_abs_time=False, use_relative_time=True,
                    use_moe=False, poly_order=1, ensure_positive_wn=True,
                    sol_net_layers=4, sol_net_hidden_dim=256, param_net_hidden_dim=128,
                    use_fourier_time=False, include_u_in_params=True,
                    traj_emb_dim=0, num_trajectories=len(pinn_train),
                    use_input_normalization=True,
                    context_hidden_dim=64, context_dim=32,
) 

ipinn_simple = SimpleIPINN(**cfg_ipinn_simple)

use_velocity_data = False
use_velocity_ic   = False

if use_velocity_data:
    print("Using position and velocity data for training.")
    data_dim = 2
else:
    print("Using position-only data for training.")
    data_dim = 1

if use_velocity_ic:
    print("Using position and velocity in initial conditions.")
    ic_dim = 2
else:
    print("Using position-only in initial conditions.")
    ic_dim = 1

# Optimizers 
optimizer_ipinn_simple  = torch.optim.AdamW(ipinn_simple.parameters(), lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_ipinn_simple  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ipinn_simple, mode="min", factor=0.5, patience=50)

writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 5_000

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch_simple_ipinn(
        model=ipinn_simple,
        residual=residual,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer=optimizer_ipinn_simple,
        criterion=criterion,
        scheduler=scheduler_ipinn_simple,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
    )

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"[Training]")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

Save

In [ ]:
# --- Save them ---
save_dir = "./models/twotank"

# Assuming your model instances are named: model_solis, model_multi, model_simple
save_model_package(solis, cfg_solis, f"{save_dir}/solis.pth")
save_model_package(ipinn_m,  cfg_ipinn_m,  f"{save_dir}/ipinn_m.pth")
save_model_package(ipinn_simple, cfg_ipinn_simple, f"{save_dir}/simple_ipinn.pth")

## Plots

### Training Set Solution & Parameter Field Learning

In [ ]:
import importlib
import solis_eval 
importlib.reload(solis_eval)
from solis_eval import eval_model, plot_surrogate_rollout

eval_model({
            "SOLIS" : solis, 
            "IPINN-M": ipinn_m, 
            "IPINN" : ipinn_simple
            }, 
        subds_train, pinn_train,
        save_path="./figures/twotank/twotank_solution.pdf",
        convert_params=True, plot_params=False,
        break_loop=True
           )

PI2NDi: 98.62532
IPINN-M: 91.248825
IPINN: 88.639145

### Parametric Solution
Online prediction

In [ ]:
test_ds = LoadedTrajectoryDataset(
    "./datasets/twotank_data_test.csv",
    traj_id_col="traj_id",
    t_col="t",
    y_cols=("y", "v"),             
    exo_cols={"u": "u"},           
    sort_by_time=True,
    device="cpu",
    dtype=torch.float32,
)

# Numerical v calculation 
for d in test_ds:
    yy = d["y"][:,0]
    tt = d["t"][:,0]
    ddy = ((yy[1:]-yy[:-1])/(tt[1:]-tt[:-1]))  # approximate velocity
    d["y"][:-1,1] = ddy
    d["y"][-1,1] = ddy[-1]  # last velocity same as second last

In [ ]:
plot_surrogate_rollout({
    "SOLIS" : solis, 
    "IPINN-M": ipinn_m, 
    "IPINN" : ipinn_simple
    }, 
    eq_pred, 
    test_ds,
    0, convert_params=True, plot_params=False,
    use_running_average_v=False,use_running_average_y=True,blend_alpha=0.05,
    save_path="./figures/twotank/twotank_rollout_alpha_005.pdf"
    )